In [ ]:
import pandas as pd
from gensim.corpora import Dictionary
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, ElectraForSequenceClassification
from nltk.sentiment import SentimentIntensityAnalyzer
import torch
import numpy as np

# Load Dataset
data = pd.read_csv('cyberbullying.csv')

# Preprocess dataset: select relevant columns and drop missing values
data = data[['tweet_text', 'cyberbullying_type']].dropna()

# Encode text labels to numeric
label_encoder = LabelEncoder()
data['label'] = label_encoder.fit_transform(data['cyberbullying_type'])

# Save the label mapping for interpretation later
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", label_mapping)

# Convert pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(data[['tweet_text', 'label']])

# Define the base models (using sklearn-compatible wrappers)
class ModelOne(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='cardiffnlp/twitter-roberta-base-sentiment'):
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_mapping))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Tokenize input and run through model
        inputs = self.tokenizer(X.tolist(), return_tensors="pt", padding=True, truncation=True, max_length=512)
        outputs = self.model(**inputs)
        return outputs.logits.detach().numpy()

class ModelTwo(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='google/electra-base-discriminator'):
        self.model = ElectraForSequenceClassification.from_pretrained(model_name, num_labels=len(label_mapping))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Tokenize input and run through model
        inputs = self.tokenizer(X.tolist(), return_tensors="pt", padding=True, truncation=True, max_length=512)
        outputs = self.model(**inputs)
        return outputs.logits.detach().numpy()

class ModelThree(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Get VADER sentiment scores
        return np.array([list(self.analyzer.polarity_scores(text).values()) for text in X])

# Vectorizer for TF-IDF model (for base models like Logistic Regression)
vectorizer = TfidfVectorizer()

# Example data (replace with actual `data['tweet_text']` and `labels`)
X_tfidf = vectorizer.fit_transform(data['tweet_text'].astype(str))
y = data['label']  # Assuming `labels` are the targets

# Split data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_tfidf, y, stratify=y)

# Define base models
base_learners = [
    ('roberta', ModelOne()),
    ('electra', ModelTwo()),
    ('vader', ModelThree())
]

# Meta-model (Logistic Regression)
meta_model = LogisticRegression()

# Stacking model (using base models and meta-model)
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_model
)

# Fit the stacking model
stacking_clf.fit(X_train, y_train)

# Predict using the stacking model
y_pred = stacking_clf.predict(X_val)

# Evaluate performance
accuracy = np.mean(y_pred == y_val)
print(f"Stacking Model Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import classification_report

# Generate classification report
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))